## This notebook uses the same pipeline multiple times but for different urls for recipes (apologies for the inconvenience)

In [ ]:
# Install dependencies (run once)
# !pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# -----------------------------
# Configuration
# -----------------------------
BASE_URL = "https://www.bbcgoodfood.com"
COLLECTION_URL = "https://www.bbcgoodfood.com/recipes/collection/quick-and-easy-family-recipes"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"
}

# -----------------------------
# Functions
# -----------------------------

def get_recipe_links(collection_url):
    """Get all recipe links from a collection page."""
    links = []
    page = 1
    while True:
        url = f"{collection_url}?page={page}"
        print(f"Fetching collection page {page}: {url}")
        res = requests.get(url, headers=headers)
        if res.status_code != 200:
            print("No more pages or failed to fetch collection.")
            break
        soup = BeautifulSoup(res.text, "html.parser")
        recipe_cards = soup.find_all("div", class_="card__section card__content")
        if not recipe_cards:
            break  # No more recipes found
        for card in recipe_cards:
            a_tag = card.find("a", class_="link d-block")
            if a_tag and a_tag.get('href'):
                href = a_tag['href']
                # Only scrape links containing /recipes/
                if "/recipes/" in href:
                    if href.startswith("http"):
                        links.append(href)
                    else:
                        links.append(BASE_URL + href)
        page += 1
        time.sleep(1)  # polite delay
    return links

def scrape_recipe(recipe_url):
    """Scrape a single recipe page for title, ingredients, and nutrition info."""
    print(f"Scraping recipe: {recipe_url}")
    res = requests.get(recipe_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    
    # Title
    title_tag = soup.find("h1", class_="post-header__title")
    title = title_tag.text.strip() if title_tag else "No title"
    
    # Ingredients
    ingredients = []
    tabbed_div = soup.find("div", class_="tabbed-list__content")
    if tabbed_div:
        ul_list = tabbed_div.find("ul", class_="ingredients-list")
        if ul_list:
            for li in ul_list.find_all("li", class_="ingredients-list__item"):
                qty_tag = li.find("span", class_="ingredients-list__item-quantity")
                ing_tag = li.find("span", class_="ingredients-list__item-ingredient")
                note_tag = li.find("div", class_="ingredients-list__item-note")
                
                qty = qty_tag.text.strip() if qty_tag else ""
                ingredient = ing_tag.text.strip() if ing_tag else ""
                note = note_tag.text.strip() if note_tag else ""
                
                ingredients.append({
                    "quantity": qty,
                    "ingredient": ingredient,
                    "note": note
                })
    
    # Nutrition
    nutrition = {}
    nutrition_list = soup.find("ul", class_="nutrition-list")
    if nutrition_list:
        for li in nutrition_list.find_all("li", class_="nutrition-list__item"):
            label_tag = li.find("span", class_="nutrition-list__label")
            if label_tag:
                label = label_tag.text.strip()
                # Some values may have extra divs, use first text after label
                value = li.get_text(separator="|").split("|")[1].strip() if "|" in li.get_text(separator="|") else ""
                nutrition[label] = value
    
    return {
        "title": title,
        "url": recipe_url,
        "ingredients": ingredients,
        "nutrition": nutrition
    }

def scrape_collection(collection_url):
    """Scrape all recipes from a collection."""
    recipe_links = get_recipe_links(collection_url)
    print(f"Found {len(recipe_links)} recipe links.")
    all_recipes = []
    
    for idx, link in enumerate(recipe_links, 1):
        try:
            recipe_data = scrape_recipe(link)
            all_recipes.append(recipe_data)
            time.sleep(1)  # polite delay
        except Exception as e:
            print(f"Failed to scrape {link}: {e}")
    
    return all_recipes

# -----------------------------
# Run the scraper
# -----------------------------
recipes = scrape_collection(COLLECTION_URL)

# -----------------------------
# Save to CSV
# -----------------------------
data_for_csv = []
for r in recipes:
    for ing in r['ingredients']:
        data_for_csv.append({
            "title": r['title'],
            "url": r['url'],
            "ingredient": ing['ingredient'],
            "quantity": ing['quantity'],
            "note": ing['note'],
            **r['nutrition']  # include all nutrition info as columns
        })

df = pd.DataFrame(data_for_csv)
df.to_csv("bbc_goodfood_recipes.csv", index=False)
print("Scraping complete! Data saved to bbc_goodfood_recipes.csv")

Fetching collection page 1: https://www.bbcgoodfood.com/recipes/collection/quick-and-easy-family-recipes?page=1
Fetching collection page 2: https://www.bbcgoodfood.com/recipes/collection/quick-and-easy-family-recipes?page=2
Fetching collection page 3: https://www.bbcgoodfood.com/recipes/collection/quick-and-easy-family-recipes?page=3
Fetching collection page 4: https://www.bbcgoodfood.com/recipes/collection/quick-and-easy-family-recipes?page=4
Fetching collection page 5: https://www.bbcgoodfood.com/recipes/collection/quick-and-easy-family-recipes?page=5
Fetching collection page 6: https://www.bbcgoodfood.com/recipes/collection/quick-and-easy-family-recipes?page=6
No more pages or failed to fetch collection.
Found 95 recipe links.
Scraping recipe: https://www.bbcgoodfood.com/recipes/creamy-chicken-pasta
Scraping recipe: https://www.bbcgoodfood.com/recipes/pasta-salmon-peas
Scraping recipe: https://www.bbcgoodfood.com/recipes/oriental-egg-fried-rice
Scraping recipe: https://www.bbcgoodfo

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"bbc_goodfood_recipes.csv")

df.head()

df["title"] = (
    df["url"]
    .str.split("/")
    .str[-1]
    .str.replace("-", " ")
    .str.title()
)

df[["title","url"]].head()

df["ingredient_full"] = (
    df["quantity"].fillna("") + " " +
    df["ingredient"].fillna("") + " " +
    df["note"].fillna("")
).str.strip()

ingredients_grouped = (
    df.groupby("url")["ingredient_full"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
)

ingredients_grouped.head()

nutrition_cols = [
    "kcal","fat","saturates","carbs",
    "sugars","fibre","protein","salt"
]

recipes = (
    df.groupby("url")
    .first()
    .reset_index()[["url","title"] + nutrition_cols]
)

recipes.head()

clean_df2 = recipes.merge(
    ingredients_grouped,
    on="url",
    how="left"
)

clean_df2.head()

clean_df2 = clean_df2.rename(columns={
    "ingredient_full":"ingredients"
})

cols = ["fat","saturates","carbs","sugars","fibre","protein","salt"]

for col in cols:
    clean_df2[col] = (
        clean_df2[col]
        .astype(str)
        .str.replace(r'(low|medium|high)', '', regex=True)  # remove labels
        .str.extract(r'([\d\.]+)')[0]                       # keep number
        .fillna('')
        + "g"                                               # add grams back
    )
    # Chat wrote this code for cleaning the dataset (title uses the last words of the url, ingredients combines quantity, ingredient and note
    # nutrition removes low/medium/high labels and keeps only numbers with 'g' unit for grams (remove random words like 16glow instead of 16g ))
clean_df2.to_csv(r"bbc_quick_and_easy_recipes.csv", index=False)

In [32]:
clean_df2.head()

,url,title,kcal,fat,saturates,carbs,sugars,fibre,protein,salt,ingredients
0,https://www.bbcgoodfood.com/recipes/15-minute-...,15 Minute Chicken Halloumi Burgers,737,42.0g,10.0g,49.0g,6.0g,4.0g,39g,4.1g,"2 chicken breast fillets, 1 tbsp oil plus extr..."
1,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Baked Potatoes,206,2.0g,0.4g,40.0g,3.0g,5.0g,5g,0.01g,"4 baking potatoes (about 250g each), ½ tbsp su..."
2,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Chicken Thighs,111,8.0g,2.0g,0.0g,0.0g,0.3g,11g,0.7g,"1 tsp paprika, ½ tsp mixed herbs, ½ tsp garlic..."
3,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Fish Fingers,64,1.0g,0.2g,7.0g,0.2g,0.3g,7g,0.7g,"250g white fish loin, 1 egg beaten, 75g panko ..."
4,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Meatballs,82,6.0g,2.0g,2.0g,0.4g,0.3g,6g,0.18g,"4 good-quality sausages, 500g beef mince, ½ on..."


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# -----------------------------
# Configuration
# -----------------------------
BASE_URL = "https://www.bbcgoodfood.com"
COLLECTION_URL = "https://www.bbcgoodfood.com/recipes/collection/batch-cooking-recipes"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# -----------------------------
# 1. Get recipe links
# -----------------------------
def get_recipe_links(collection_url):
    links = []
    page = 1

    while True:
        url = f"{collection_url}?page={page}"
        print(f"Fetching page {page}...")

        res = requests.get(url, headers=headers)
        if res.status_code != 200:
            break

        soup = BeautifulSoup(res.text, "html.parser")

        cards = soup.find_all("div", class_="card__section card__content")

        if not cards:
            break

        for card in cards:
            a = card.find("a", href=True)
            if a:
                href = a["href"]
                if "/recipes/" in href:
                    full_url = href if href.startswith("http") else BASE_URL + href
                    if full_url not in links:
                        links.append(full_url)

        page += 1
        time.sleep(1)

    return links


# -----------------------------
# 2. Scrape each recipe
# -----------------------------
def scrape_recipe(url):
    print(f"Scraping: {url}")

    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    # ---- TITLE ----
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else "No title"

    # ---- INGREDIENTS ----
    ingredients = []
    for li in soup.select("li.ingredients-list__item"):
        qty = li.select_one(".ingredients-list__item-quantity")
        ing = li.select_one(".ingredients-list__item-ingredient")
        note = li.select_one(".ingredients-list__item-note")

        quantity = qty.get_text(strip=True) if qty else ""
        ingredient = ing.get_text(strip=True) if ing else ""
        note_text = note.get_text(strip=True) if note else ""

        ingredients.append({
            "quantity": quantity,
            "ingredient": ingredient,
            "note": note_text
        })

    # ---- NUTRITION ----
    nutrition = {}
    for li in soup.select("ul.nutrition-list li"):
        label = li.select_one(".nutrition-list__label")
        if label:
            key = label.get_text(strip=True)
            value = li.get_text(strip=True).replace(key, "").strip()
            nutrition[key] = value

    return {
        "title": title,
        "url": url,
        "ingredients": ingredients,
        "nutrition": nutrition
    }


# -----------------------------
# 3. Scrape full collection
# -----------------------------
def scrape_collection(collection_url):
    recipe_links = get_recipe_links(collection_url)
    print(f"\nFound {len(recipe_links)} recipes\n")

    all_data = []

    for i, link in enumerate(recipe_links, 1):
        try:
            data = scrape_recipe(link)
            all_data.append(data)
            print(f"[{i}/{len(recipe_links)}] Done: {data['title']}")
            time.sleep(1)
        except Exception as e:
            print(f"Error on {link}: {e}")

    return all_data


# -----------------------------
# 4. Run scraper
# -----------------------------
recipes = scrape_collection(COLLECTION_URL)

# -----------------------------
# 5. Convert to CSV
# -----------------------------
rows = []

for r in recipes:
    for ing in r["ingredients"]:
        row = {
            "title": r["title"],
            "url": r["url"],
            "ingredient": ing["ingredient"],
            "quantity": ing["quantity"],
            "note": ing["note"]
        }

        # Add nutrition fields dynamically
        for k, v in r["nutrition"].items():
            row[k] = v

        rows.append(row)

df = pd.DataFrame(rows)

In [ ]:
# Save the already scraped DataFrame safely

# Option 1: save to your folder (fixed path)
df = df.to_csv(r"bbc_batch_cooking_recipes.csv", index=False)

print("✅ File saved successfully!")

✅ File saved successfully!


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"bbc_batch_cooking_recipes.csv")

df.head()

df["title"] = (
    df["url"]
    .str.split("/")
    .str[-1]
    .str.replace("-", " ")
    .str.title()
)

df[["title","url"]].head()

df["ingredient_full"] = (
    df["quantity"].fillna("") + " " +
    df["ingredient"].fillna("") + " " +
    df["note"].fillna("")
).str.strip()

ingredients_grouped = (
    df.groupby("url")["ingredient_full"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
)

ingredients_grouped.head()

nutrition_cols = [
    "kcal","fat","saturates","carbs",
    "sugars","fibre","protein","salt"
]

recipes = (
    df.groupby("url")
    .first()
    .reset_index()[["url","title"] + nutrition_cols]
)

recipes.head()

clean_df = recipes.merge(
    ingredients_grouped,
    on="url",
    how="left"
)

clean_df.head()

clean_df = clean_df.rename(columns={
    "ingredient_full":"ingredients"
})

cols = ["fat","saturates","carbs","sugars","fibre","protein","salt"]

for col in cols:
    clean_df[col] = (
        clean_df[col]
        .astype(str)
        .str.replace(r'(low|medium|high)', '', regex=True)  # remove labels
        .str.extract(r'([\d\.]+)')[0]                       # keep number
        .fillna('')
        + "g"                                               # add grams back
    )
    # Chat wrote this code for cleaning the dataset (title uses the last words of the url, ingredients combines quantity, ingredient and note
    # nutrition removes low/medium/high labels and keeps only numbers with 'g' unit for grams (remove random words like 16glow instead of 16g ))
clean_df.to_csv("bbc_batch_cooking_recipes.csv", index=False)
clean_df.head()

,url,title,kcal,fat,saturates,carbs,sugars,fibre,protein,salt,ingredients
0,https://www.bbcgoodfood.com/recipes/apple-pear...,Apple Pear Cherry Compote,199,1g,0g,51g,47g,4g,1g,0.03g,"8 eating apples peeled, cored and cut into chu..."
1,https://www.bbcgoodfood.com/recipes/aubergine-...,Aubergine Parmigiana Lasagne,466,23g,9g,41g,12g,7g,20g,0.6g,3 large aubergines trimmed and thinly sliced l...
2,https://www.bbcgoodfood.com/recipes/balsamic-s...,Balsamic Steaks Peppercorn Wedges,534,25g,8g,43g,4g,5g,34g,0.3g,"2 tbsp balsamic vinegar, 2 tsp concentrated li..."
3,https://www.bbcgoodfood.com/recipes/basic-lentils,Basic Lentils,230,5g,3g,31g,4g,5g,13g,0.1g,"2 tbsp coconut oil, 2 onions chopped, 4 garlic..."
4,https://www.bbcgoodfood.com/recipes/bean-sausa...,Bean Sausage Hotpot,474,23g,8g,39g,10g,9g,29g,3.55g,8 large sausages (course-cut ones like Toulous...


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# -----------------------------
# Configuration
# -----------------------------
BASE_URL = "https://www.bbcgoodfood.com"
COLLECTION_URL = "https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"
}

# -----------------------------
# Functions
# -----------------------------
def get_recipe_links(collection_url):
    """Get all recipe links from a collection page."""
    links = []
    page = 1
    while True:
        url = f"{collection_url}?page={page}"
        print(f"Fetching collection page {page}: {url}")
        res = requests.get(url, headers=headers)
        if res.status_code != 200:
            print("No more pages or failed to fetch collection.")
            break
        soup = BeautifulSoup(res.text, "html.parser")
        recipe_cards = soup.find_all("div", class_="card__section card__content")
        if not recipe_cards:
            break  # No more recipes found
        for card in recipe_cards:
            a_tag = card.find("a", class_="link d-block")
            if a_tag and a_tag.get('href'):
                href = a_tag['href']
                if "/recipes/" in href:
                    if href.startswith("http"):
                        links.append(href)
                    else:
                        links.append(BASE_URL + href)
        page += 1
        time.sleep(1)
    return links

def scrape_recipe(recipe_url):
    """Scrape a single recipe page for title, ingredients, and nutrition info."""
    print(f"Scraping recipe: {recipe_url}")
    res = requests.get(recipe_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    
    # Title
    title_tag = soup.find("h1", class_="post-header__title")
    title = title_tag.text.strip() if title_tag else "No title"
    
    # Ingredients
    ingredients = []
    tabbed_div = soup.find("div", class_="tabbed-list__content")
    if tabbed_div:
        ul_list = tabbed_div.find("ul", class_="ingredients-list")
        if ul_list:
            for li in ul_list.find_all("li", class_="ingredients-list__item"):
                qty_tag = li.find("span", class_="ingredients-list__item-quantity")
                ing_tag = li.find("span", class_="ingredients-list__item-ingredient")
                note_tag = li.find("div", class_="ingredients-list__item-note")
                qty = qty_tag.text.strip() if qty_tag else ""
                ingredient = ing_tag.text.strip() if ing_tag else ""
                note = note_tag.text.strip() if note_tag else ""
                ingredients.append({"quantity": qty, "ingredient": ingredient, "note": note})
    
    # Nutrition
    nutrition = {}
    nutrition_list = soup.find("ul", class_="nutrition-list")
    if nutrition_list:
        for li in nutrition_list.find_all("li", class_="nutrition-list__item"):
            label_tag = li.find("span", class_="nutrition-list__label")
            if label_tag:
                label = label_tag.text.strip()
                value = li.get_text(separator="|").split("|")[1].strip() if "|" in li.get_text(separator="|") else ""
                nutrition[label] = value
    
    return {"title": title, "url": recipe_url, "ingredients": ingredients, "nutrition": nutrition}

def scrape_collection(collection_url):
    """Scrape all recipes from a collection."""
    recipe_links = get_recipe_links(collection_url)
    print(f"Found {len(recipe_links)} recipe links.")
    all_recipes = []
    for idx, link in enumerate(recipe_links, 1):
        try:
            recipe_data = scrape_recipe(link)
            all_recipes.append(recipe_data)
            time.sleep(1)
        except Exception as e:
            print(f"Failed to scrape {link}: {e}")
    return all_recipes

# -----------------------------
# Run the scraper
# -----------------------------
recipes = scrape_collection(COLLECTION_URL)

# -----------------------------
# Save to CSV
# -----------------------------
data_for_csv = []
for r in recipes:
    for ing in r['ingredients']:
        data_for_csv.append({
            "title": r['title'],
            "url": r['url'],
            "ingredient": ing['ingredient'],
            "quantity": ing['quantity'],
            "note": ing['note'],
            **r['nutrition']
        })

df = pd.DataFrame(data_for_csv)
df.to_csv(r"dinner_recipes.csv", index=False)
print("Scraping complete! Data saved to dinner_recipes.csv")

Fetching collection page 1: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=1
Fetching collection page 2: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=2
Fetching collection page 3: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=3
Fetching collection page 4: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=4
Fetching collection page 5: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=5
Fetching collection page 6: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=6
Fetching collection page 7: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=7
Fetching collection page 8: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=8
No more pages or failed to fetch collection.
Found 142 recipe links.
Scraping recipe: https://www.bbcgoodfood.com/recipes/chicken-pasta-bake
Scraping recipe: https://www.bbcgoodfood.co

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"dinner_recipes.csv")

df.head()

df["title"] = (
    df["url"]
    .str.split("/")
    .str[-1]
    .str.replace("-", " ")
    .str.title()
)

df[["title","url"]].head()

df["ingredient_full"] = (
    df["quantity"].fillna("") + " " +
    df["ingredient"].fillna("") + " " +
    df["note"].fillna("")
).str.strip()

ingredients_grouped = (
    df.groupby("url")["ingredient_full"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
)

ingredients_grouped.head()

nutrition_cols = [
    "kcal","fat","saturates","carbs",
    "sugars","fibre","protein","salt"
]

recipes = (
    df.groupby("url")
    .first()
    .reset_index()[["url","title"] + nutrition_cols]
)

recipes.head()

clean_df = recipes.merge(
    ingredients_grouped,
    on="url",
    how="left"
)

clean_df.head()

clean_df = clean_df.rename(columns={
    "ingredient_full":"ingredients"
})

cols = ["fat","saturates","carbs","sugars","fibre","protein","salt"]

for col in cols:
    clean_df[col] = (
        clean_df[col]
        .astype(str)
        .str.replace(r'(low|medium|high)', '', regex=True)  # remove labels
        .str.extract(r'([\d\.]+)')[0]                       # keep number
        .fillna('')
        + "g"                                               # add grams back
    )
    # Chat wrote this code for cleaning the dataset (title uses the last words of the url, ingredients combines quantity, ingredient and note
    # nutrition removes low/medium/high labels and keeps only numbers with 'g' unit for grams (remove random words like 16glow instead of 16g ))
clean_df.to_csv("dinner_recipes.csv", index=False)
clean_df.head()

,url,title,kcal,fat,saturates,carbs,sugars,fibre,protein,salt,ingredients
0,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Crispy Chilli Beef,440.0,19.0g,3.0g,34g,20.0g,3.0g,31.0g,3.1g,250g thin-cut minute steak thinly sliced into ...
1,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Sweet Potato Jackets With Tahini Yogurt,581.0,30.0g,9.0g,59g,34.0g,1.0g,12.0g,0.4g,"2 sweet potatoes, 1 tbsp olive oil plus 2 tsp,..."
2,https://www.bbcgoodfood.com/recipes/aubergine-...,Aubergine Halloumi Harissa Skillet Bake,546.0,39.0g,20.0g,15g,14.0g,7.0g,31.0g,3.5g,"2-4 tbsp olive oil plus a drizzle, 1 large aub..."
3,https://www.bbcgoodfood.com/recipes/baked-salmon,Baked Salmon,354.0,23.0g,4.0g,1g,0.2g,0.4g,35.0g,0.1g,"4 skinless salmon fillets, 1 tbsp olive oil or..."
4,https://www.bbcgoodfood.com/recipes/bean-hallo...,Bean Halloumi Stew,468.0,25.0g,7.0g,36g,12.0g,14.0g,19.0g,1.9g,"3 tbsp olive oil, 1 onion thinly sliced, 1 red..."


In [ ]:
# Install dependencies (if not already)
# !pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# -----------------------------
# Configuration
# -----------------------------
BASE_URL = "https://www.bbcgoodfood.com"
COLLECTION_URL = "https://www.bbcgoodfood.com/recipes/collection/breakfast-recipes"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"
}

# -----------------------------
# Functions
# -----------------------------
def get_recipe_links(collection_url):
    """Get all recipe links from a collection page."""
    links = []
    page = 1
    while True:
        url = f"{collection_url}?page={page}"
        print(f"Fetching collection page {page}: {url}")
        res = requests.get(url, headers=headers)
        if res.status_code != 200:
            print("No more pages or failed to fetch collection.")
            break
        soup = BeautifulSoup(res.text, "html.parser")
        recipe_cards = soup.find_all("div", class_="card__section card__content")
        if not recipe_cards:
            break  # No more recipes found
        for card in recipe_cards:
            a_tag = card.find("a", class_="link d-block")
            if a_tag and a_tag.get('href'):
                href = a_tag['href']
                if "/recipes/" in href:
                    if href.startswith("http"):
                        links.append(href)
                    else:
                        links.append(BASE_URL + href)
        page += 1
        time.sleep(1)
    return links

def scrape_recipe(recipe_url):
    """Scrape a single recipe page for title, ingredients, and nutrition info."""
    print(f"Scraping recipe: {recipe_url}")
    res = requests.get(recipe_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    
    # Title
    title_tag = soup.find("h1", class_="post-header__title")
    title = title_tag.text.strip() if title_tag else "No title"
    
    # Ingredients
    ingredients = []
    tabbed_div = soup.find("div", class_="tabbed-list__content")
    if tabbed_div:
        ul_list = tabbed_div.find("ul", class_="ingredients-list")
        if ul_list:
            for li in ul_list.find_all("li", class_="ingredients-list__item"):
                qty_tag = li.find("span", class_="ingredients-list__item-quantity")
                ing_tag = li.find("span", class_="ingredients-list__item-ingredient")
                note_tag = li.find("div", class_="ingredients-list__item-note")
                qty = qty_tag.text.strip() if qty_tag else ""
                ingredient = ing_tag.text.strip() if ing_tag else ""
                note = note_tag.text.strip() if note_tag else ""
                ingredients.append({"quantity": qty, "ingredient": ingredient, "note": note})
    
    # Nutrition
    nutrition = {}
    nutrition_list = soup.find("ul", class_="nutrition-list")
    if nutrition_list:
        for li in nutrition_list.find_all("li", class_="nutrition-list__item"):
            label_tag = li.find("span", class_="nutrition-list__label")
            if label_tag:
                label = label_tag.text.strip()
                value = li.get_text(separator="|").split("|")[1].strip() if "|" in li.get_text(separator="|") else ""
                nutrition[label] = value
    
    return {"title": title, "url": recipe_url, "ingredients": ingredients, "nutrition": nutrition}

def scrape_collection(collection_url):
    """Scrape all recipes from a collection."""
    recipe_links = get_recipe_links(collection_url)
    print(f"Found {len(recipe_links)} recipe links.")
    all_recipes = []
    for idx, link in enumerate(recipe_links, 1):
        try:
            recipe_data = scrape_recipe(link)
            all_recipes.append(recipe_data)
            time.sleep(1)
        except Exception as e:
            print(f"Failed to scrape {link}: {e}")
    return all_recipes

# -----------------------------
# Run the scraper
# -----------------------------
recipes = scrape_collection(COLLECTION_URL)

# -----------------------------
# Save to CSV
# -----------------------------
data_for_csv = []
for r in recipes:
    for ing in r['ingredients']:
        data_for_csv.append({
            "title": r['title'],
            "url": r['url'],
            "ingredient": ing['ingredient'],
            "quantity": ing['quantity'],
            "note": ing['note'],
            **r['nutrition']
        })

df = pd.DataFrame(data_for_csv)
df.to_csv(r"breakfast_recipes.csv", index=False)
print("Scraping complete! Data saved to breakfast.csv")

Fetching collection page 1: https://www.bbcgoodfood.com/recipes/collection/breakfast-recipes?page=1
Fetching collection page 2: https://www.bbcgoodfood.com/recipes/collection/breakfast-recipes?page=2
Fetching collection page 3: https://www.bbcgoodfood.com/recipes/collection/breakfast-recipes?page=3
Fetching collection page 4: https://www.bbcgoodfood.com/recipes/collection/breakfast-recipes?page=4
Fetching collection page 5: https://www.bbcgoodfood.com/recipes/collection/breakfast-recipes?page=5
Fetching collection page 6: https://www.bbcgoodfood.com/recipes/collection/breakfast-recipes?page=6
No more pages or failed to fetch collection.
Found 96 recipe links.
Scraping recipe: https://www.bbcgoodfood.com/recipes/american-pancakes
Scraping recipe: https://www.bbcgoodfood.com/recipes/the-breakfast-club
Scraping recipe: https://www.bbcgoodfood.com/recipes/pain-au-chocolat
Scraping recipe: https://www.bbcgoodfood.com/recipes/leek-kale-hash-with-sage-fried-eggs
Scraping recipe: https://www.b

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"breakfast_recipes.csv")

df.head()

df["title"] = (
    df["url"]
    .str.split("/")
    .str[-1]
    .str.replace("-", " ")
    .str.title()
)

df[["title","url"]].head()

df["ingredient_full"] = (
    df["quantity"].fillna("") + " " +
    df["ingredient"].fillna("") + " " +
    df["note"].fillna("")
).str.strip()

ingredients_grouped = (
    df.groupby("url")["ingredient_full"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
)

ingredients_grouped.head()

nutrition_cols = [
    "kcal","fat","saturates","carbs",
    "sugars","fibre","protein","salt"
]

recipes = (
    df.groupby("url")
    .first()
    .reset_index()[["url","title"] + nutrition_cols]
)

recipes.head()

clean_df = recipes.merge(
    ingredients_grouped,
    on="url",
    how="left"
)

clean_df.head()

clean_df = clean_df.rename(columns={
    "ingredient_full":"ingredients"
})

cols = ["fat","saturates","carbs","sugars","fibre","protein","salt"]

for col in cols:
    clean_df[col] = (
        clean_df[col]
        .astype(str)
        .str.replace(r'(low|medium|high)', '', regex=True)  # remove labels
        .str.extract(r'([\d\.]+)')[0]                       # keep number
        .fillna('')
        + "g"                                               # add grams back
    )
    # Chat wrote this code for cleaning the dataset (title uses the last words of the url, ingredients combines quantity, ingredient and note
    # nutrition removes low/medium/high labels and keeps only numbers with 'g' unit for grams (remove random words like 16glow instead of 16g ))
clean_df.to_csv("breakfast_recipes.csv", index=False)
clean_df.head()

,url,title,kcal,fat,saturates,carbs,sugars,fibre,protein,salt,ingredients
0,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Bacon,74,6g,2.0g,0.0g,0.0g,0.0g,6.0g,0.88g,6 rashers streaky bacon or 3 rashers back bacon
1,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Fry Up,845,44g,14.0g,61.0g,7.0g,9.0g,47.0g,4.1g,"1 English muffin, ½ tsp butter softened, 200g ..."
2,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Omelette,251,22g,10.0g,0.1g,0.1g,0.0g,13.0g,0.68g,"15g butter, 2 eggs, fillings of your choice, s..."
3,https://www.bbcgoodfood.com/recipes/american-p...,American Pancakes,356,13g,6.0g,46.0g,8.0g,2.0g,13.0g,1.3g,"200g self-raising flour, 1 ½ tsp baking powder..."
4,https://www.bbcgoodfood.com/recipes/avocado-to...,Avocado Toast Chorizo Fried Eggs,522,37g,10.0g,25.0g,3.0g,4.0g,23.0g,1.5g,"1 tbsp pumpkin seed, 85g chorizo sliced into c..."


In [ ]:
# Install dependencies (if not already)
# !pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# -----------------------------
# Configuration
# -----------------------------
BASE_URL = "https://www.bbcgoodfood.com"
COLLECTION_URL = "https://www.bbcgoodfood.com/recipes/collection/quick-lunch-recipes"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"
}

# -----------------------------
# Functions
# -----------------------------
def get_recipe_links(collection_url):
    """Get all recipe links from a collection page."""
    links = []
    page = 1
    while True:
        url = f"{collection_url}?page={page}"
        print(f"Fetching collection page {page}: {url}")
        res = requests.get(url, headers=headers)
        if res.status_code != 200:
            print("No more pages or failed to fetch collection.")
            break
        soup = BeautifulSoup(res.text, "html.parser")
        recipe_cards = soup.find_all("div", class_="card__section card__content")
        if not recipe_cards:
            break  # No more recipes found
        for card in recipe_cards:
            a_tag = card.find("a", class_="link d-block")
            if a_tag and a_tag.get('href'):
                href = a_tag['href']
                if "/recipes/" in href:
                    if href.startswith("http"):
                        links.append(href)
                    else:
                        links.append(BASE_URL + href)
        page += 1
        time.sleep(1)
    return links

def scrape_recipe(recipe_url):
    """Scrape a single recipe page for title, ingredients, and nutrition info."""
    print(f"Scraping recipe: {recipe_url}")
    res = requests.get(recipe_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    
    # Title
    title_tag = soup.find("h1", class_="post-header__title")
    title = title_tag.text.strip() if title_tag else "No title"
    
    # Ingredients
    ingredients = []
    tabbed_div = soup.find("div", class_="tabbed-list__content")
    if tabbed_div:
        ul_list = tabbed_div.find("ul", class_="ingredients-list")
        if ul_list:
            for li in ul_list.find_all("li", class_="ingredients-list__item"):
                qty_tag = li.find("span", class_="ingredients-list__item-quantity")
                ing_tag = li.find("span", class_="ingredients-list__item-ingredient")
                note_tag = li.find("div", class_="ingredients-list__item-note")
                qty = qty_tag.text.strip() if qty_tag else ""
                ingredient = ing_tag.text.strip() if ing_tag else ""
                note = note_tag.text.strip() if note_tag else ""
                ingredients.append({"quantity": qty, "ingredient": ingredient, "note": note})
    
    # Nutrition
    nutrition = {}
    nutrition_list = soup.find("ul", class_="nutrition-list")
    if nutrition_list:
        for li in nutrition_list.find_all("li", class_="nutrition-list__item"):
            label_tag = li.find("span", class_="nutrition-list__label")
            if label_tag:
                label = label_tag.text.strip()
                value = li.get_text(separator="|").split("|")[1].strip() if "|" in li.get_text(separator="|") else ""
                nutrition[label] = value
    
    return {"title": title, "url": recipe_url, "ingredients": ingredients, "nutrition": nutrition}

def scrape_collection(collection_url):
    """Scrape all recipes from a collection."""
    recipe_links = get_recipe_links(collection_url)
    print(f"Found {len(recipe_links)} recipe links.")
    all_recipes = []
    for idx, link in enumerate(recipe_links, 1):
        try:
            recipe_data = scrape_recipe(link)
            all_recipes.append(recipe_data)
            time.sleep(1)
        except Exception as e:
            print(f"Failed to scrape {link}: {e}")
    return all_recipes

# -----------------------------
# Run the scraper
# -----------------------------
recipes = scrape_collection(COLLECTION_URL)

# -----------------------------
# Save to CSV
# -----------------------------
data_for_csv = []
for r in recipes:
    for ing in r['ingredients']:
        data_for_csv.append({
            "title": r['title'],
            "url": r['url'],
            "ingredient": ing['ingredient'],
            "quantity": ing['quantity'],
            "note": ing['note'],
            **r['nutrition']
        })

df = pd.DataFrame(data_for_csv)
df.to_csv(r"lunch_recipes.csv", index=False)
print("Scraping complete! Data saved to lunch_recipes.csv")

Fetching collection page 1: https://www.bbcgoodfood.com/recipes/collection/quick-lunch-recipes?page=1
Fetching collection page 2: https://www.bbcgoodfood.com/recipes/collection/quick-lunch-recipes?page=2
Fetching collection page 3: https://www.bbcgoodfood.com/recipes/collection/quick-lunch-recipes?page=3
Fetching collection page 4: https://www.bbcgoodfood.com/recipes/collection/quick-lunch-recipes?page=4
Fetching collection page 5: https://www.bbcgoodfood.com/recipes/collection/quick-lunch-recipes?page=5
No more pages or failed to fetch collection.
Found 76 recipe links.
Scraping recipe: https://www.bbcgoodfood.com/recipes/falafel-burgers-0
Scraping recipe: https://www.bbcgoodfood.com/recipes/quick-chicken-hummus-bowl
Scraping recipe: https://www.bbcgoodfood.com/recipes/10minute-couscous-salad
Scraping recipe: https://www.bbcgoodfood.com/recipes/smashed-peas-on-toast
Scraping recipe: https://www.bbcgoodfood.com/recipes/spicy-chicken-avocado-wraps
Scraping recipe: https://www.bbcgoodfoo

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"lunch_recipes.csv")

df.head()

df["title"] = (
    df["url"]
    .str.split("/")
    .str[-1]
    .str.replace("-", " ")
    .str.title()
)

df[["title","url"]].head()

df["ingredient_full"] = (
    df["quantity"].fillna("") + " " +
    df["ingredient"].fillna("") + " " +
    df["note"].fillna("")
).str.strip()

ingredients_grouped = (
    df.groupby("url")["ingredient_full"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
)

ingredients_grouped.head()

nutrition_cols = [
    "kcal","fat","saturates","carbs",
    "sugars","fibre","protein","salt"
]

recipes = (
    df.groupby("url")
    .first()
    .reset_index()[["url","title"] + nutrition_cols]
)

recipes.head()

clean_df = recipes.merge(
    ingredients_grouped,
    on="url",
    how="left"
)

clean_df.head()

clean_df = clean_df.rename(columns={
    "ingredient_full":"ingredients"
})

cols = ["fat","saturates","carbs","sugars","fibre","protein","salt"]

for col in cols:
    clean_df[col] = (
        clean_df[col]
        .astype(str)
        .str.replace(r'(low|medium|high)', '', regex=True)  # remove labels
        .str.extract(r'([\d\.]+)')[0]                       # keep number
        .fillna('')
        + "g"                                               # add grams back
    )
    # Chat wrote this code for cleaning the dataset (title uses the last words of the url, ingredients combines quantity, ingredient and note
    # nutrition removes low/medium/high labels and keeps only numbers with 'g' unit for grams (remove random words like 16glow instead of 16g ))
clean_df.to_csv("lunch_recipes.csv", index=False)
clean_df.head()

,url,title,kcal,fat,saturates,carbs,sugars,fibre,protein,salt,ingredients
0,https://www.bbcgoodfood.com/recipes/10minute-c...,10Minute Couscous Salad,468,24g,5g,44.0g,7.0g,5g,16g,1.23g,"100g couscous, 200ml hot low salt vegetable st..."
1,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Cheese Ham Toastie,627,38g,22g,43.0g,3.0g,5g,27g,2.88g,"20g butter softened, 2 slices sourdough or oth..."
2,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Ham Cheese Egg Bagel,505,17g,7g,56.0g,13.0g,2g,30g,2.7g,"1 bagel halved, ½ tbsp chilli jam or red onion..."
3,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Mushrooms On Toast,349,8g,1g,53.0g,4.0g,4g,15g,1.6g,"300g chestnut mushrooms quartered or sliced, a..."
4,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Omelette,251,22g,10g,0.1g,0.1g,0g,13g,0.68g,"15g butter, 2 eggs, fillings of your choice, s..."
